# tetrad_port Demo

This notebook demonstrates causal discovery via the `tetrad_port` Python package, which wraps a C++ implementation of CMU's Tetrad library.

**Supported algorithms:** PC, FGES, GFCI, BOSS, BOSS-FCI, GRaSP, GRaSP-FCI

The workflow:
1. Load/generate data as a pandas DataFrame
2. Optionally prepare data (standardize, add lags)
3. Run causal discovery (via `tp.run()` or algorithm-specific methods)
4. Inspect discovered edges
5. Fit a SEM model to estimate edge weights
6. Visualize the causal graph

In [ ]:
import pandas as pd
import numpy as np
from tetrad_port import TetradPort

tp = TetradPort(verbose=False)
print(f"tetrad_port version: {TetradPort.__module__}")

import tetrad_port
print(f"Version: {tetrad_port.__version__}")

## 1. Generate Synthetic Data

We create data from a known DAG: **X → Y → Z ← W**, with X also causing Z (a diamond + collider structure).

```
X → Y
X → Z ← W
Y → Z
```

In [ ]:
np.random.seed(42)
n = 2000

X = np.random.randn(n)
W = np.random.randn(n)
Y = 0.7 * X + 0.5 * np.random.randn(n)
Z = 0.5 * X + 0.5 * Y + 0.6 * W + 0.3 * np.random.randn(n)

df = pd.DataFrame({'X': X, 'Y': Y, 'Z': Z, 'W': W})
print(f"Data shape: {df.shape}")
df.head()

## 2. Run the PC Algorithm

Use `tp.run("pc", ...)` (dispatcher) or `tp.run_pc(...)` (direct method).

In [ ]:
# Using the run() dispatcher (equivalent to tp.run_pc(...))
results, graph_info = tp.run(df, algorithm="pc", alpha=0.05)

print(f"Discovered {results['num_edges']} edges among {results['num_nodes']} nodes:\n")
for edge in results['edges']:
    print(f"  {edge}")

## 3. Inspect Graph Structure

In [ ]:
print("Directed edges:")
for src, dst in graph_info['directed_edges']:
    print(f"  {src} → {dst}")

print("\nUndirected edges:")
for n1, n2 in graph_info['undirected_edges']:
    print(f"  {n1} — {n2}")

print("\nAdjacency list:")
for node, neighbors in sorted(graph_info['adjacency'].items()):
    if neighbors:
        print(f"  {node}: {', '.join(sorted(neighbors))}")

## 4. SEM Fitting for Edge Weights

Convert the discovered edges to a lavaan model and fit with semopy to get standardized path coefficients.

In [ ]:
lavaan_model = tp.edges_to_lavaan(results['edges'])
print("Lavaan model:")
print(lavaan_model)

In [ ]:
try:
    sem_results = tp.run_semopy(lavaan_model, df)
    print("SEM Parameter Estimates:\n")
    display(sem_results['estimates'])
except ImportError as e:
    print(f"SEM fitting requires semopy: {e}")
    print("Install with: pip install tetrad-port[sem]")

## 5. Data Preparation Helpers

For time-series data, `TetradPort` provides helpers matching the FastCDA pattern.

In [ ]:
# Standardize
df_std = tp.standardize_df_cols(df)
print("After standardization:")
print(f"  Means: {df_std.mean().round(10).to_dict()}")
print(f"  Stds:  {df_std.std().round(2).to_dict()}")

In [ ]:
# Add lag columns (for time-series causal analysis)
df_lagged = tp.add_lag_columns(df, n_lags=1)
print(f"Original columns: {list(df.columns)}")
print(f"After lagging:    {list(df_lagged.columns)}")
print(f"Shape: {df.shape} → {df_lagged.shape}")

In [ ]:
# Generate temporal knowledge for lagged data
# Returns a Knowledge object directly (pass as_dict=True for a plain dict)
knowledge = tp.create_lag_knowledge(list(df.columns))
print(f"Knowledge type: {type(knowledge).__name__}")

# Can also pass a dict directly to any run method:
knowledge_dict = tp.create_lag_knowledge(list(df.columns), as_dict=True)
print(f"\nDict form: {knowledge_dict}")

## 6. Example with Collider Structure

The PC algorithm correctly identifies v-structures (colliders). Here we generate data from **X → Z ← Y** where X and Y are independent.

In [ ]:
np.random.seed(123)
n = 2000
X = np.random.randn(n)
Y = np.random.randn(n)
Z = 0.8 * X + 0.8 * Y + 0.3 * np.random.randn(n)

df_collider = pd.DataFrame({'X': X, 'Y': Y, 'Z': Z})
results_c, graph_c = tp.run_pc(df_collider, alpha=0.05)

print(f"Edges ({results_c['num_edges']}):")
for edge in results_c['edges']:
    print(f"  {edge}")

print(f"\nDirected: {graph_c['directed_edges']}")
print(f"Undirected: {graph_c['undirected_edges']}")

## 7. Visualization with DgraphFlex (Optional)

If `dgraph_flex` is installed, you can visualize the causal graph just like in FastCDA.

In [ ]:
try:
    from dgraph_flex import DgraphFlex

    dg = DgraphFlex()
    dg.add_edges(results['edges'])
    dg.show_graph()
except ImportError:
    print("DgraphFlex not installed. Install with: pip install dgraph-flex")
    print("\nEdge list (text representation):")
    for edge in results['edges']:
        print(f"  {edge}")

## 8. Loading Your Own Data

Replace the synthetic data with your own CSV:

In [ ]:
# df_real = pd.read_csv("your_data.csv")
# df_real_std = tp.standardize_df_cols(df_real)
# results, graph_info = tp.run_pc(df_real_std, alpha=0.01, depth=3)
# for edge in results['edges']:
#     print(edge)